In [1]:
%load_ext autoreload
%autoreload 2
import NuLattice.one_body_operators as obops
import NuLattice.two_body_operators as tbops
import NuLattice.lattice as lat
import NuLattice.FCI.few_body_diagonalization as fbd
from scipy.sparse.linalg import eigsh as arpack_eigsh

In [2]:
thisL = 4
a = 1.0 / 100.0
lattice = lat.get_lattice(thisL)

In [3]:
myTkin=obops.tKin(thisL, 3, a)
print("number of matrix elements from kinetic energy", len(myTkin))

number of matrix elements from kinetic energy 2560


In [4]:
#INTERACTION A
sNL = 0.077
cNL = -0.2268
cINL = 0.02184
bpi = 0.7

In [5]:
v_OPE = tbops.onePionEx(thisL, bpi, a, lattice, verbose=True)

Calculating f_SS...Done
Calculating Densities...Done
Generating Interaction...
Compressing interaction...Done
Interaction Generated


In [6]:
v_NL=tbops.shortRangeV_2body(lattice, thisL, 0, sNL, cNL, a, verbose=True, max_mem=1e8)
iso_ops = [2.0 * obops.list_to_sparse1b(obops.tau_x(lattice, thisL)), 
           2.0 * obops.list_to_sparse1b(obops.tau_y(lattice, thisL)), 
           2.0 * obops.list_to_sparse1b(obops.tau_z(lattice, thisL))]
for tau_I in iso_ops:
    v_NL += tbops.shortRangeV_2body(lattice, thisL, 0, sNL, cINL, a, op1b=tau_I,verbose=True, max_mem=1e8)

Generating Densities...Done
Generating Interaction...Interaction Generated
Generating Densities...Done
Generating Interaction...Interaction Generated
Generating Densities...Done
Generating Interaction...Interaction Generated
Generating Densities...Done
Generating Interaction...Interaction Generated


In [7]:
mycontactA = tbops.sparse_to_list_2body(v_NL+v_OPE, thisL)
print("number of matrix elements from two-body contacts", len(mycontactA))

number of matrix elements from two-body contacts 1352848


In [8]:
print("Computing deuteron w/ Interaction A")
# additive quantum numbers
numpart=2 # number of nucleons
tz = 0    # twice the value of isospin projection
sz = None    # twice the value of spin projection
my_basis = lat.get_sp_basis(thisL)
nstat =  len(my_basis)
# get two-body basis as a dictionary for lookup
H2_lookup = fbd.get_many_body_states(my_basis, numpart, total_tz=tz, total_sz=sz)
print("matrix dimension:", len(H2_lookup))

# make scipy.sparse.csr_matrix for kinetic energy 
T2_csr_mat = fbd.get_csr_matrix_scalar_op(H2_lookup, myTkin, nstat)
print("kinetic energy done")

# make scipy.sparse.csr_matrix for 2-body interactions 
V2_csr_mat = fbd.get_csr_matrix_scalar_op(H2_lookup, mycontactA, nstat)
print("2-body interaction done")

# add all into Hamiltonian
H2_csr_mat = T2_csr_mat + V2_csr_mat

# compute lowest eigenvalue(s)
k_eig=2  # number of eigenvalues
vals, vecs = arpack_eigsh(H2_csr_mat, k=k_eig, which='SA')
print("Energies (MeV):", vals)

Computing deuteron w/ Interaction A
matrix dimension: 16384
kinetic energy done
2-body interaction done
Energies (MeV): [-5.02030089 -5.02030153]


In [9]:
#INTERACTION B
sNL = 0.077
cNL = -0.1171
cINL = 0.02607
sL = 0.81
cL = -0.01013
cSL = - cL / 3
cIL = cSL
cSIL = cSL

In [10]:
v_NL=tbops.shortRangeV_2body(lattice, thisL, 0, sNL, cNL, a, verbose=True, max_mem=1e8)
iso_ops = [2.0 * obops.list_to_sparse1b(obops.tau_x(lattice, thisL)), 
           2.0 * obops.list_to_sparse1b(obops.tau_y(lattice, thisL)), 
           2.0 * obops.list_to_sparse1b(obops.tau_z(lattice, thisL))]
for tau_I in iso_ops:
    v_NL += tbops.shortRangeV_2body(lattice, thisL, 0, sNL, cINL, a, op1b=tau_I,verbose=True, max_mem=1e8)

Generating Densities...Done
Generating Interaction...Interaction Generated
Generating Densities...Done
Generating Interaction...Interaction Generated
Generating Densities...Done
Generating Interaction...Interaction Generated
Generating Densities...Done
Generating Interaction...Interaction Generated


In [11]:
v_L = tbops.shortRangeV_2body(lattice, thisL, sL, 0, cL, a,verbose=True,max_mem=1e8)
sp_ops = [  2.0 * obops.list_to_sparse1b(obops.spin_x(lattice, thisL)), 
            2.0 * obops.list_to_sparse1b(obops.spin_y(lattice, thisL)), 
            2.0 * obops.list_to_sparse1b(obops.spin_z(lattice, thisL))]
for tau_I in iso_ops:
    v_L += tbops.shortRangeV_2body(lattice, thisL, sL, 0, cIL, a, op1b=tau_I, verbose=True,max_mem=1e8)
    for sigma_S in sp_ops:
        v_L += tbops.shortRangeV_2body(lattice, thisL, sL, 0, cSIL, a, op1b=sigma_S @ tau_I, verbose=True, max_mem=1e8)
for sigma_S in sp_ops:
        v_L += tbops.shortRangeV_2body(lattice, thisL, sL, 0, cSL, a, op1b=sigma_S, verbose=True, max_mem=1e8)

Generating Densities...Done
Performing Local Smearing...Done
Generating Interaction...Interaction Generated
Generating Densities...Done
Performing Local Smearing...Done
Generating Interaction...Interaction Generated
Generating Densities...Done
Performing Local Smearing...Done
Generating Interaction...Interaction Generated
Generating Densities...Done
Performing Local Smearing...Done
Generating Interaction...Interaction Generated
Generating Densities...Done
Performing Local Smearing...Done
Generating Interaction...Interaction Generated
Generating Densities...Done
Performing Local Smearing...Done
Generating Interaction...Interaction Generated
Generating Densities...Done
Performing Local Smearing...Done
Generating Interaction...Interaction Generated
Generating Densities...Done
Performing Local Smearing...Done
Generating Interaction...Interaction Generated
Generating Densities...Done
Performing Local Smearing...Done
Generating Interaction...Interaction Generated
Generating Densities...Done


In [12]:
mycontactB = tbops.sparse_to_list_2body(v_L+v_NL+v_OPE, thisL)
print("number of matrix elements from two-body contacts", len(mycontactB))

number of matrix elements from two-body contacts 1376016


In [13]:
print("Computing deuteron w/ Interaction B")
# make scipy.sparse.csr_matrix for 2-body interactions 
V2_csr_mat = fbd.get_csr_matrix_scalar_op(H2_lookup, mycontactB, nstat)
print("2-body interaction done")

# add all into Hamiltonian
H2_csr_mat = T2_csr_mat + V2_csr_mat

# compute lowest eigenvalue(s)
k_eig=2  # number of eigenvalues
vals, vecs = arpack_eigsh(H2_csr_mat, k=k_eig, which='SA')
print("Energies (MeV):", vals)

Computing deuteron w/ Interaction B
2-body interaction done
Energies (MeV): [-5.3978004  -5.39779987]
